# Memory Experiment: Chunked Vocabulary Output & Fused Loss (Soup Kernel Challenge)

Kaggle settings (right panel):
- **Accelerator:** GPU T4 x2
- **Internet:** On (required for Hugging Face and package installation)

This notebook runs the experimental memory-optimization evaluation on branch `memory-experiments`:
1. **Environment & Dependency Audit:** Verifies T4 GPU and pinned package versions.
2. **Repository Checkout:** Clones branch `memory-experiments`.
3. **Gate 1 (Unit Tests):** Executes full custom kernel unit test suite (`pytest solution -v`), including the new chunked-loss test suite.
4. **Gate 1B (RoPE GPU & PTX Diagnostic):** Runs RoPE on CUDA, validates FP16/FP32 precision and PTX instructions.
5. **Gate 1C (Chunked Causal LM Loss Diagnostic):** Verifies chunked causal loss, hidden gradients, streaming entropy/accuracy, and rewind row losses.
6. **Gate 2 (Qwen2MLP Equivalence):** Compares output and input gradients against PyTorch reference with frozen weights.
7. **Gate 3 (Soup Regression Suite):** Runs official 79 tests with `SOUP_APPLY_SOLUTION=1`.
8. **Benchmark:** Executes official benchmark `run_task.py --repeats 2` measuring wall time and peak memory.
9. **Scoring & Rule Evaluation:** Runs `check_rules.py` and `score.py`.

In [ ]:
# Memory experiment repository and branch configuration
REPO_URL = "https://github.com/raulibaddin/soup-kernel-challenge-demo.git"
BRANCH = "memory-experiments"
COMMIT = None  # Use latest HEAD of memory-experiments

In [ ]:
import os, platform, subprocess, sys

# 1. Hardware verification
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print("Detected GPU:", gpus.strip())
assert "T4" in gpus, "Accelerator must be GPU T4 x2"

# 2. Token setup
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

# 3. Pinned dependency installation matching competition specification
print("\nInstalling official pinned dependencies...")
install_cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "soup-cli[train]==0.77.0",
    "triton==3.6.0",
    "transformers==5.16.1",
    "peft==0.20.0",
    "trl==0.29.1",
    "bitsandbytes==0.50.2",
    "accelerate==1.14.0",
    "pytest"
]
subprocess.run(install_cmd, check=True)

# 4. Strict environment and package version verification
from importlib.metadata import version
import torch
import triton
import transformers
import peft
import trl
import bitsandbytes as bnb
import accelerate

soup_version = version("soup-cli")

print("\n=== Environment & Dependency Version Audit ===")
print("Python:       ", platform.python_version())
print("PyTorch:      ", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version: ", torch.version.cuda)
print("Device Name:  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
print("Device Count: ", torch.cuda.device_count())
print("Triton:       ", getattr(triton, "__version__", "unknown"))
print("Soup CLI:     ", soup_version)
print("Transformers: ", transformers.__version__)
print("PEFT:         ", peft.__version__)
print("TRL:          ", trl.__version__)
print("BitsAndBytes: ", bnb.__version__)
print("Accelerate:   ", accelerate.__version__)

# Verify official competition environment and pinned package requirements
assert (3, 11) <= sys.version_info[:2] < (3, 14), (
    f"Soup CLI 0.77.0 requires Python 3.11–3.13, "
    f"got {platform.python_version()}"
)
assert torch.cuda.is_available() and torch.version.cuda is not None, "CUDA-enabled PyTorch is required"
assert "T4" in torch.cuda.get_device_name(0), f"Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
assert soup_version == "0.77.0", f"Expected soup-cli==0.77.0, got {soup_version}"
assert getattr(triton, "__version__", "").split("+")[0] == "3.6.0", f"Expected triton==3.6.0, got {getattr(triton, '__version__', None)}"
assert transformers.__version__ == "5.16.1", f"Expected transformers==5.16.1, got {transformers.__version__}"
assert peft.__version__ == "0.20.0", f"Expected peft==0.20.0, got {peft.__version__}"
assert trl.__version__ == "0.29.1", f"Expected trl==0.29.1, got {trl.__version__}"
assert bnb.__version__ == "0.50.2", f"Expected bitsandbytes==0.50.2, got {bnb.__version__}"
assert accelerate.__version__ == "1.14.0", f"Expected accelerate==1.14.0, got {accelerate.__version__}"
torch_major_minor = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
assert torch_major_minor >= (2, 6), f"soup-cli train requires torch>=2.6.0 compatibility floor, got {torch.__version__}"
print("All official competition dependencies and pinned requirements verified successfully!")

In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("Cloning without GITHUB_TOKEN secret:", type(exc).__name__)

auth = []
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]

shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
clone_cmd = ["git", *auth, "clone", "-q", "-b", BRANCH, REPO_URL, "/kaggle/working/repo"]
clone = subprocess.run(clone_cmd, capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)

if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)

current_commit = subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                                capture_output=True, text=True).stdout
print("Active Repository Commit:", current_commit.strip())

In [ ]:
import subprocess
# Prepare task dataset
subprocess.run(["python", "task/prepare_data.py"], cwd="/kaggle/working/repo", check=True)

# Gate 1: Run custom kernel & loss unit tests
print("Running full test suite including chunked loss (pytest solution -v)...")
proc_gate1 = subprocess.run(["python", "-m", "pytest", "solution", "-v"], cwd="/kaggle/working/repo")
gate1_rc = proc_gate1.returncode
print(f"\nGate 1 (pytest solution) exit code: {gate1_rc}")
if gate1_rc != 0:
    print("WARNING: Custom unit tests failed; proceeding to diagnostics before gating.")

In [ ]:
# Gate 1B: Focused RoPE GPU PTX and precision diagnostic on CUDA
rope_diag = '''
import torch, sys
sys.path.insert(0, '/kaggle/working/repo')
from solution.rope import apply_triton_rope, rope_fwd_kernel, rope_bwd_kernel
from solution.test_kernels import _reference_rope

print("=== RoPE GPU Diagnostic & PTX Inspection ===")

try:
    B, H, S, D = 4, 12, 512, 128
    HALF_D = D // 2
    x_test = torch.empty((B * H * S, D), device='cuda', dtype=torch.float16)
    cos_test = torch.empty((B * S, D), device='cuda', dtype=torch.float16)
    sin_test = torch.empty((B * S, D), device='cuda', dtype=torch.float16)
    out_test = torch.empty_like(x_test)
    grid = (B * H * S,)
    compiled = rope_fwd_kernel[grid](
        x_test, cos_test, sin_test, out_test,
        B * H * S, S, H * S,
        HAS_B_COS=True, D=D, HALF_D=HALF_D
    )
    ptx = compiled.asm.get('ptx', '')
    if ptx:
        fma_f16 = 'fma.rn.f16' in ptx or 'hfma2' in ptx
        fma_f32 = 'fma.rn.f32' in ptx
        mul_f16 = 'mul.rn.f16' in ptx or 'hmul2' in ptx
        add_f16 = 'add.rn.f16' in ptx or 'hadd2' in ptx
        print(f"PTX Inspection: fma.rn.f16/hfma2={fma_f16}, fma.rn.f32={fma_f32}, mul.f16/hmul2={mul_f16}, add.f16/hadd2={add_f16}")
    else:
        print("PTX inspection: asm dict did not contain 'ptx' key.")
except Exception as exc:
    print(f"PTX inspection note: {exc}")

tol_fp16 = 2e-3
tol_fp32 = 1e-4
failures = []

def audit_case(name, q, cos, sin, grad, tol):
    ref_q, _ = _reference_rope(q, q, cos, sin)
    ref_q.backward(grad)
    ref_dq = q.grad.clone()

    q_tri = q.detach().clone().requires_grad_(True)
    tri_q = apply_triton_rope(q_tri, cos, sin)
    tri_q.backward(grad)

    diff_fwd = (tri_q.float() - ref_q.float().detach()).abs()
    diff_bwd = (q_tri.grad.float() - ref_dq.float()).abs()
    
    thresh_fwd = tol + tol * ref_q.float().detach().abs()
    thresh_bwd = tol + tol * ref_dq.float().abs()

    mismatch_fwd = (diff_fwd > thresh_fwd).sum().item()
    mismatch_bwd = (diff_bwd > thresh_bwd).sum().item()
    total = diff_fwd.numel()

    max_abs_fwd = diff_fwd.max().item()
    max_rel_fwd = (diff_fwd / (ref_q.float().detach().abs() + 1e-5)).max().item()
    max_abs_bwd = diff_bwd.max().item()
    max_rel_bwd = (diff_bwd / (ref_dq.float().abs() + 1e-5)).max().item()

    status_fwd = 'PASS' if mismatch_fwd == 0 else 'FAIL'
    status_bwd = 'PASS' if mismatch_bwd == 0 else 'FAIL'
    overall = 'PASS' if (mismatch_fwd == 0 and mismatch_bwd == 0) else 'FAIL'
    
    print(f"[{overall}] {name}:")
    print(f"  Forward:  {status_fwd} ({mismatch_fwd}/{total} mismatches, max_abs={max_abs_fwd:.2e}, max_rel={max_rel_fwd:.2e})")
    print(f"  Backward: {status_bwd} ({mismatch_bwd}/{total} mismatches, max_abs={max_abs_bwd:.2e}, max_rel={max_rel_bwd:.2e})")
    
    if overall == 'FAIL':
        failures.append((name, mismatch_fwd, mismatch_bwd))

# Test 1: Prior failing FP16 shape
torch.manual_seed(42)
B, H, S, D = 4, 12, 512, 128
q = torch.randn((B, H, S, D), device='cuda', dtype=torch.float16, requires_grad=True)
cos = torch.randn((B, S, D), device='cuda', dtype=torch.float16)
sin = torch.randn((B, S, D), device='cuda', dtype=torch.float16)
grad = torch.randn((B, H, S, D), device='cuda', dtype=torch.float16)
audit_case('Prior failing FP16 shape (4, 12, 512, 128)', q, cos, sin, grad, tol_fp16)

# Test 2: Realistic frequency embeddings
inv_freq = 1.0 / (1000000.0 ** (torch.arange(0, D, 2, device='cuda', dtype=torch.float32) / D))
t = torch.arange(S, device='cuda', dtype=torch.float32)
freqs = torch.outer(t, inv_freq)
emb = torch.cat([freqs, freqs], dim=-1)
cos_real = emb.cos().unsqueeze(0).expand(B, -1, -1).to(torch.float16)
sin_real = emb.sin().unsqueeze(0).expand(B, -1, -1).to(torch.float16)
q_real = torch.randn((B, H, S, D), device='cuda', dtype=torch.float16, requires_grad=True)
grad_real = torch.randn((B, H, S, D), device='cuda', dtype=torch.float16)
audit_case('Realistic frequencies FP16 (4, 12, 512, 128)', q_real, cos_real, sin_real, grad_real, tol_fp16)

# Test 3: FP32 shape
q_fp32 = torch.randn((B, H, S, D), device='cuda', dtype=torch.float32, requires_grad=True)
cos_fp32 = torch.randn((B, S, D), device='cuda', dtype=torch.float32)
sin_fp32 = torch.randn((B, S, D), device='cuda', dtype=torch.float32)
grad_fp32 = torch.randn((B, H, S, D), device='cuda', dtype=torch.float32)
audit_case('FP32 standard shape (4, 12, 512, 128)', q_fp32, cos_fp32, sin_fp32, grad_fp32, tol_fp32)

if failures:
    print()
    print(f"RoPE GPU diagnostic recorded {len(failures)} failure(s):")
    for name, f_fwd, f_bwd in failures:
        print(f"  - {name}: {f_fwd} fwd mismatches, {f_bwd} bwd mismatches")
    sys.exit(1)
else:
    print()
    print("All RoPE GPU equivalence and runtime dtype checks passed successfully!")
'''
proc_gate1b = subprocess.run(["python", "-c", rope_diag], cwd="/kaggle/working/repo")
gate1b_rc = proc_gate1b.returncode
print(f"\nGate 1B (RoPE GPU diagnostic) exit code: {gate1b_rc}")

In [ ]:
# Gate 1C: Chunked Causal LM Loss Diagnostic on CUDA
chunk_diag = '''
import torch, sys
import torch.nn.functional as F
sys.path.insert(0, '/kaggle/working/repo')
from solution.chunk_loss import chunked_causal_lm_loss, CHUNKED_LOSS_CALL_COUNT

print("=== Chunked Causal LM Loss Diagnostic on CUDA ===")

torch.manual_seed(42)
B, S, H, V = 4, 128, 512, 4096
hidden_ref = torch.randn((B, S, H), device='cuda', dtype=torch.float16, requires_grad=True)
hidden_chk = hidden_ref.clone().detach().requires_grad_(True)

lm_head = torch.nn.Linear(H, V, bias=False, device='cuda', dtype=torch.float16)
for p in lm_head.parameters():
    p.requires_grad_(False)

labels = torch.randint(0, V, (B, S), device='cuda')
labels[0, :8] = -100
labels[1, 60:] = -100

# 1. Unchunked PyTorch reference
logits_full = lm_head(hidden_ref)
shift_logits = logits_full[:, :-1, :].reshape(-1, V)
shift_labels = labels[:, 1:].reshape(-1)
loss_ref = F.cross_entropy(shift_logits.float(), shift_labels, ignore_index=-100)
loss_ref.backward()
dx_ref = hidden_ref.grad.clone()

# 2. Custom chunked implementation
count_before = CHUNKED_LOSS_CALL_COUNT
loss_chk, metrics = chunked_causal_lm_loss(hidden_chk, lm_head.weight, labels, chunk_size=32)
loss_chk.backward()
dx_chk = hidden_chk.grad.clone()
from solution.chunk_loss import CHUNKED_LOSS_CALL_COUNT as count_after

loss_diff = abs(loss_ref.item() - loss_chk.item())
dx_diff = (dx_ref - dx_chk).abs().max().item()
print(f"Loss Reference: {loss_ref.item():.6f}, Chunked: {loss_chk.item():.6f}, Diff: {loss_diff:.2e}")
print(f"Hidden dx Max Diff: {dx_diff:.2e}")
print(f"Chunked Loss Invocation Count: {count_before} -> {count_after}")

assert loss_diff < 1e-3, f"Loss diff {loss_diff} exceeds threshold"
assert dx_diff < 1e-3, f"Hidden state grad diff {dx_diff} exceeds threshold"
assert count_after > count_before, "Chunked loss path did not execute"

# 3. Verify per-row losses match rewind log formula
token_loss_2d = metrics.token_loss[:, :-1]
tokens_calc = (labels[:, 1:] != -100).sum(dim=1)
totals_calc = token_loss_2d.sum(dim=1)
row_losses = [float(tot / tok.clamp(min=1)) for tot, tok in zip(totals_calc, tokens_calc)]
print(f"Calculated per-row losses for rewind: {row_losses}")

print("Chunked Causal LM Loss CUDA diagnostic passed successfully!")
'''
proc_gate1c = subprocess.run(["python", "-c", chunk_diag], cwd="/kaggle/working/repo")
gate1c_rc = proc_gate1c.returncode
print(f"\nGate 1C (Chunked Loss diagnostic) exit code: {gate1c_rc}")

In [ ]:
# Gate Check: Halt validation if preliminary kernel or loss tests failed
preliminary_failures = []
if 'gate1_rc' not in globals() or gate1_rc != 0:
    preliminary_failures.append(f"Gate 1 (pytest solution) failed with exit code {globals().get('gate1_rc', 'NOT_RUN')}")
if 'gate1b_rc' not in globals() or gate1b_rc != 0:
    preliminary_failures.append(f"Gate 1B (RoPE GPU diagnostic) failed with exit code {globals().get('gate1b_rc', 'NOT_RUN')}")
if 'gate1c_rc' not in globals() or gate1c_rc != 0:
    preliminary_failures.append(f"Gate 1C (Chunked Loss diagnostic) failed with exit code {globals().get('gate1c_rc', 'NOT_RUN')}")
if preliminary_failures:
    raise RuntimeError(f"Validation stopped at preliminary gate: {'; '.join(preliminary_failures)}")
print("Preliminary kernel and loss verification gates (1, 1B, 1C) PASSED. Proceeding to integration.")

In [ ]:
# Gate 2: Qwen2MLP Integration & Equivalence Verification
mlp_diag = '''
import torch, sys
sys.path.insert(0, '/kaggle/working/repo')
from transformers.models.qwen2.modeling_qwen2 import Qwen2MLP, Qwen2Config
from solution.swiglu import patch_qwen2_mlp

print("=== Qwen2MLP Forward & Backward Integration Diagnostic ===")
config = Qwen2Config(
    hidden_size=1536,
    intermediate_size=8960,
    hidden_act="silu",
)
torch.manual_seed(42)
mlp_ref = Qwen2MLP(config).cuda().half()
mlp_custom = Qwen2MLP(config).cuda().half()
mlp_custom.load_state_dict(mlp_ref.state_dict())

# In the competition task, base MLP weights are frozen (LoRA targets q_proj, v_proj)
for model in (mlp_ref, mlp_custom):
    for param in model.parameters():
        param.requires_grad_(False)

B, S, H = 4, 512, 1536
x_ref = torch.randn((B, S, H), device='cuda', dtype=torch.float16, requires_grad=True)
x_custom = x_ref.detach().clone().requires_grad_(True)
grad_out = torch.randn((B, S, H), device='cuda', dtype=torch.float16)

# 1. Unpatched reference forward & backward
out_ref = mlp_ref(x_ref)
out_ref.backward(grad_out)

# 2. Apply patch and run custom forward & backward
patch_qwen2_mlp()
out_custom = mlp_custom(x_custom)
out_custom.backward(grad_out)

tol = 2e-3
diff_out = (out_custom.float() - out_ref.float()).abs()
diff_dx = (x_custom.grad.float() - x_ref.grad.float()).abs()

thresh_out = tol + tol * out_ref.float().abs()
thresh_dx = tol + tol * x_ref.grad.float().abs()

mismatch_out = (diff_out > thresh_out).sum().item()
mismatch_dx = (diff_dx > thresh_dx).sum().item()
total = diff_out.numel()

print(f"MLP Forward Output:     {'PASS' if mismatch_out == 0 else 'FAIL'} ({mismatch_out}/{total} mismatches, max_abs={diff_out.max().item():.2e})")
print(f"MLP Input Gradient (dx): {'PASS' if mismatch_dx == 0 else 'FAIL'} ({mismatch_dx}/{total} mismatches, max_abs={diff_dx.max().item():.2e})")

if mismatch_out > 0 or mismatch_dx > 0:
    print()
    print("Qwen2MLP integration test failed!")
    sys.exit(1)
else:
    print()
    print("Qwen2MLP integration test passed successfully!")
'''
proc_gate2 = subprocess.run(["python", "-c", mlp_diag], cwd="/kaggle/working/repo", check=True)

In [ ]:
import os, subprocess
# Gate 3: Official Soup regression test suite (enforcing SOUP_APPLY_SOLUTION=1)
print("Running official Soup regression test suite with solution enabled...")
env = dict(os.environ, SOUP_APPLY_SOLUTION="1")
subprocess.run(["python", "-m", "pytest", "tests", "-q"], cwd="/kaggle/working/repo", env=env, check=True)

In [ ]:
import subprocess
# Step 4: Full official training benchmark (2 repeats: base, sol, sol, base)
print("Running full benchmark harness with 2 repeats...")
subprocess.run(["python", "harness/run_task.py", "--repeats", "2"], cwd="/kaggle/working/repo", check=True)

In [ ]:
import os, subprocess, sys

# Step 5: Official rule checks and score calculation
print("=== Running check_rules.py ===")
proc_check = subprocess.run(["python", "harness/check_rules.py"], cwd="/kaggle/working/repo")
check_rc = proc_check.returncode

print("\n=== Running score.py ===")
result_json = "/kaggle/working/repo/runs/result.json"
result_exists = os.path.exists(result_json)
score_rc = None
if result_exists:
    proc_score = subprocess.run(["python", "harness/score.py", result_json], cwd="/kaggle/working/repo")
    score_rc = proc_score.returncode
else:
    print(f"score.py skipped: {result_json} does not exist (run_task failed before creating results).")

errors = []
if check_rc != 0:
    errors.append(f"check_rules.py failed (exit code {check_rc})")
if not result_exists:
    errors.append(f"{result_json} was not produced")
elif score_rc is not None and score_rc != 0:
    errors.append(f"score.py failed (exit code {score_rc})")

if errors:
    raise RuntimeError("Validation failed at Step 5: " + "; ".join(errors))

print("\n=== Full Validation & Scoring Completed Successfully ===")